# Text-to-SQL Agent — Testing Notebook

Tests the **DuckDB text-to-SQL agent** (`src/agents/sql_analyst.py`), an explicit LangGraph pipeline:

```
generate ─▶ execute ─▶ [judge] ─▶ respond ─▶ check
(reasoning   (DuckDB     optional   (shared    (grounding
 + SQL)       runs it)   reviewer)   Responder)  check)
    ▲            │           │
    └── retry on error / judge feedback
```

- **All arithmetic in SQL** — the model never computes numbers itself.
- **Responder** (`src/agents/responder.py`) writes the answer a little warmer, but only
  from figures it's given.
- **Grounding check** (`src/checks.py`) — deterministic; flags any answer number not
  backed by the data.
- **LLM judge** — optional evaluator-optimizer, toggled by `USE_JUDGE` in config/.env.

`ask_sql` returns: **reasoning, sql, answer, data, grounded, unsupported, judge_ok, judge_feedback**.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

## The graph

In [2]:
from src.agents.sql_analyst import sql_agent

print(sql_agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	execute(execute)
	judge(judge)
	respond(respond)
	check(check)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	execute -.-> generate;
	execute -.-> judge;
	execute -.-> respond;
	generate -.-> execute;
	generate -.-> respond;
	judge -.-> generate;
	judge -.-> respond;
	respond --> check;
	check --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Part 1 — DuckDB layer & safety guard (no API calls)

The ledger parquet is a read-only DuckDB view called `ledger`. Only SELECT/WITH runs.

In [3]:
from src.agents.sql_analyst import _is_safe, _execute_sql

checks = {
    "SELECT * FROM ledger": True,
    "DROP TABLE ledger": False,
    "SELECT 1; DROP TABLE ledger": False,   # stacked statements
    "COPY ledger TO 'out.csv'": False,       # filesystem write
}
for q, expected in checks.items():
    print(f"{_is_safe(q)!s:5}  (expected {expected!s:5})  {q}")

True   (expected True )  SELECT * FROM ledger
False  (expected False)  DROP TABLE ledger
False  (expected False)  SELECT 1; DROP TABLE ledger
False  (expected False)  COPY ledger TO 'out.csv'


In [4]:
_execute_sql("SELECT ledger_type, ROUND(SUM(profit), 2) AS pnl FROM ledger GROUP BY ledger_type")

{'data': [{'ledger_type': 'expenses', 'pnl': -1354321.02},
  {'ledger_type': 'revenue', 'pnl': 2887652.89}]}

In [5]:
# A broken query returns the SQL error (the agent uses this to self-correct)
_execute_sql("SELECT nope FROM ledger")

{'error': 'SQL error: Binder Error: Referenced column "nope" not found in FROM clause!\nCandidate bindings: "property_name"\n\nLINE 1: SELECT nope FROM ledger\n               ^'}

## Part 2 — Grounding check (deterministic, no API calls)

`check_grounding` confirms every meaningful number in an answer came from the data.
It resolves shorthand ($99.5K → 99500) and ignores structural integers (list ranks).

In [6]:
from src.checks import check_grounding

data = [{"current": 361810.32, "prior": 262309.07, "difference": 99501.25}]

# 1) A faithful answer -> grounded
print("faithful:  ", check_grounding("Profit rose from $262,309.07 to $361,810.32.", data)["grounded"])

# 2) Shorthand still resolves -> grounded
print("shorthand: ", check_grounding("A difference of about $99.5K.", data)["grounded"])

# 3) A fabricated / miscalculated number -> caught
bad = check_grounding("Profit was $500,000.00.", data)
print("fabricated:", bad["grounded"], "-> unsupported:", bad["unsupported"])

# 4) Structural numbers (ranks) are ignored
print("structural:", check_grounding("Top 3: 1. A  2. B  3. C ($361,810.32)", data)["grounded"])

faithful:   True
shorthand:  True
fabricated: False -> unsupported: [500000.0]
structural: True


## Part 3 — The LLM judge (optional evaluator-optimizer)

Off by default (`USE_JUDGE`); enable by setting `USE_JUDGE=true` in `.env`. Here we
exercise the judge component directly to show it discriminates good vs. bad SQL.

In [7]:
from src.config import USE_JUDGE
from src.prompts import SQL_JUDGE_PROMPT
from src.agents.sql_analyst import _judge
from langchain_core.messages import HumanMessage, SystemMessage

print("USE_JUDGE (from config):", USE_JUDGE)


def judge(question, reasoning, sql, results):
    verdict = _judge.invoke([SystemMessage(SQL_JUDGE_PROMPT),
                             HumanMessage(f"Question: {question}\nReasoning: {reasoning}\n"
                                          f"SQL: {sql}\nResults (sample): {results}")])
    print(f"is_good={verdict.is_good}  feedback={verdict.feedback!r}")

USE_JUDGE (from config): False


In [8]:
# A correct query -> approved
judge("Total P&L in 2024?", "Sum profit for 2024.",
      "SELECT SUM(profit) AS pnl FROM ledger WHERE year = '2024'",
      [{"pnl": 1171521.55}])

is_good=True  feedback=''


In [9]:
# A wrong query (ignores the year filter) -> rejected with feedback
judge("Total P&L in 2024?", "Sum all profit.",
      "SELECT SUM(profit) AS pnl FROM ledger",
      [{"pnl": 1533331.87}])

is_good=False  feedback='The query is missing a WHERE clause to filter for 2024 data only; it currently sums profit across all years in the ledger table.'


## Part 4 — The full agent (judge off by default)

`show()` prints the reasoning, the SQL, the answer, and the grounding result.

In [10]:
from src.agents.sql_analyst import ask_sql


def show(question: str):
    r = ask_sql(question)
    print("Q:", question)
    print("\nREASONING:", r["reasoning"])
    print("\nSQL:", " ".join(r["sql"].split()) if r["sql"] else "(none — not answerable)")
    print("\nANSWER:", r["answer"])
    print(f"\nGROUNDED: {r['grounded']}   unsupported: {r['unsupported']}   judge_ok: {r['judge_ok']}")
    print("=" * 80)

### Straightforward and compound questions

In [11]:
show("What is the total P&L for all properties in 2024?")

Q: What is the total P&L for all properties in 2024?

REASONING: The question asks for the total P&L (profit and loss) for all properties in 2024. The ledger table contains a 'profit' column (net; revenue positive, expenses negative) and a 'year' column. I can sum all profit values where year = '2024' to get the total P&L for that year.

SQL: SELECT SUM(profit) AS total_pl_2024 FROM ledger WHERE year = '2024';

ANSWER: The total P&L for all properties in 2024 is **$1,171,521.55**.

GROUNDED: True   unsupported: []   judge_ok: None


In [12]:
show("Who are my top 3 tenants by profit?")

Q: Who are my top 3 tenants by profit?

REASONING: The ledger table contains tenant_name and profit columns. I can aggregate profit by tenant_name and order by profit descending to identify the top 3 tenants. I'll exclude NULL tenant_name rows (entity-level lines) and return the top 3.

SQL: SELECT tenant_name, SUM(profit) AS total_profit FROM ledger WHERE tenant_name IS NOT NULL GROUP BY tenant_name ORDER BY total_profit DESC LIMIT 3;

ANSWER: Your top 3 tenants by profit are:

1. **Tenant 7** – $880,512.18
2. **Tenant 14** – $391,490.29
3. **Tenant 11** – $292,531.00

GROUNDED: True   unsupported: []   judge_ok: None


In [13]:
show("How does Q1 2025 compare to Q1 2024 in profit, and what drove the difference?")

Q: How does Q1 2025 compare to Q1 2024 in profit, and what drove the difference?

REASONING: The ledger table contains profit data by quarter and ledger_type (revenue vs. expenses). I can compare Q1 2025 to Q1 2024 by filtering on quarter values and aggregating profit. To identify what drove the difference, I'll break down the comparison by ledger_type (revenue and expenses) so we can see which component changed.

SQL: SELECT ledger_type, SUM(profit) FILTER (WHERE quarter = '2025-Q1') AS q1_2025, SUM(profit) FILTER (WHERE quarter = '2024-Q1') AS q1_2024, SUM(profit) FILTER (WHERE quarter = '2025-Q1') - SUM(profit) FILTER (WHERE quarter = '2024-Q1') AS difference, ROUND(100.0 * (SUM(profit) FILTER (WHERE quarter = '2025-Q1') - SUM(profit) FILTER (WHERE quarter = '2024-Q1')) / ABS(SUM(profit) FILTER (WHERE quarter = '2024-Q1')), 2) AS pct_change FROM ledger GROUP BY ledger_type UNION ALL SELECT 'TOTAL' AS ledger_type, SUM(profit) FILTER (WHERE quarter = '2025-Q1') AS q1_2025, SUM(profit)

### Robustness — out-of-scope, unknown, relative time

In [14]:
show("What is the market price of Building 17?")

Q: What is the market price of Building 17?

REASONING: The ledger table contains only financial transactions (revenue and expenses) recorded as profit entries. It does not contain asset valuations, market prices, or property appraisals. Market price is external valuation data that would not be found in an accounting ledger.

SQL: (none — not answerable)

ANSWER: I'm unable to answer that question from the available financial data. The ledger contains only transaction records—revenue and expenses—and doesn't include asset valuations or market prices. To find the market price of Building 17, you'd need to consult property appraisals, market valuation reports, or external real estate databases rather than the accounting ledger.

GROUNDED: True   unsupported: []   judge_ok: None


In [15]:
show("What is the P&L for Building 999?")

Q: What is the P&L for Building 999?

REASONING: The question asks for the P&L (profit and loss) for Building 999. The ledger table contains a 'profit' column where revenue is positive and expenses are negative. I can sum the profit for all rows where property_name = 'Building 999' to get the net P&L for that building.

SQL: SELECT SUM(profit) AS net_pl FROM ledger WHERE property_name = 'Building 999';

ANSWER: I don't have a P&L figure for Building 999. The results show that the net P&L data is not available (appears as missing/null in the system).

To get this information, you may need to:
- Verify the building ID is correct
- Check if P&L data has been computed and loaded for this property
- Contact your data team if the building exists but figures haven't been calculated yet

GROUNDED: True   unsupported: []   judge_ok: None


In [16]:
show("What was my profit last quarter?")

Q: What was my profit last quarter?

REASONING: The question asks for profit in the last quarter. Given that the latest data is 2025-M03 (which falls in 2025-Q1), the "last quarter" refers to 2024-Q4. I can sum the profit column filtered by quarter = '2024-Q4' to get the total profit for that quarter.

SQL: SELECT SUM(profit) AS profit_last_quarter FROM ledger WHERE quarter = '2024-Q4';

ANSWER: Your profit last quarter was **$278,954.87**.

GROUNDED: True   unsupported: []   judge_ok: None


## Notes

- **Responder split:** answer-writing lives in `src/agents/responder.py`, runs a bit
  warmer, and only reports figures it is given — reused by every lane in the full graph.
- **Grounding check** (`src/checks.py`) is deterministic and cheap; it caught the
  fabricated `$500,000` above while passing faithful answers and shorthand.
- **Judge** is an optional evaluator-optimizer: enable with `USE_JUDGE=true`; it can send
  a bad query back to `generate` with feedback before any answer is written.
- Relative-time and *vague*-question handling will move up to the Router agent, next.